In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root      43579 F...m python3
/dev/nvidiactl:      root      43579 F...m python3
/dev/nvidia-uvm:     root      43579 F...m python3


In [2]:
# Clear filesystem cache (pagecache, dentries, inodes)
!sudo sync
!sudo sh -c 'echo 3 > /proc/sys/vm/drop_caches'

sh: 1: cannot create /proc/sys/vm/drop_caches: Read-only file system


In [3]:
# Check GPU status
!nvidia-smi

Sat Feb 28 21:45:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             29W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Data configuration
DATA_ID = 'jxie/coco_captions'
DATA_SPLIT = 'train'
TRAIN_SIZE = 1000
TEST_SIZE = 200

In [5]:
from datasets import load_dataset, Dataset

def load_hf_dataset(
    data_id,
    split='train',
    train_size = 1000,
    test_size = 200,
):
    # Use streaming
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Manually take train_size + test_size samples
    total_size = train_size + test_size
    sliced_data = []
    
    dataset_size = 0
    unique_cocoids = set()
    
    for example in dataset_stream:
        if dataset_size >= total_size:
            break
        
        # Ensure unique cocoids if cocoid exists
        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)
        
        sliced_data.append(example)
        dataset_size += 1

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset

dataset = load_hf_dataset(DATA_ID, split=DATA_SPLIT, train_size=TRAIN_SIZE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

In [6]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

if 'url' in dataset.features:
    dataset = dataset.map(process_image_with_url, num_proc=4) # Load and process images from URLs
else:
    dataset = dataset.map(process_image, num_proc=4) # Resize to (512, 512) and convert to RGB

Map (num_proc=4):   0%|          | 0/1200 [00:00<?, ? examples/s]

In [7]:
# Remove the original 'image' column
train_dataset = dataset.remove_columns('image')

# Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column('decoded_image', 'image')

In [10]:
from huggingface_hub import hf_hub_download

K = 8192

codebook_path = hf_hub_download(
    repo_id='alxxtexxr/BIVR-Data',
    filename=f'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/codebook_faiss.{K}.npy',
    repo_type='dataset',
)

print("Codebook downloaded to:", codebook_path)

model_unsloth_Qwen2.5-VL-7B-Instruct-bnb(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BIVR-Data/snapshots/4e290fa44d1501c6b7b4c69b826c03770af9bb3a/model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train/codebook_faiss.8192.npy


In [11]:
%%capture
%pip install faiss-gpu-cu12

In [12]:
import numpy as np
import faiss

codebook = np.load(codebook_path).astype(np.float32) # (K, 2048) -> current: (16_384, 2048)
faiss.normalize_L2(codebook)

print("Codebook shape:", codebook.shape)
print("Mean L2 norm (should be ~1):", np.linalg.norm(codebook, axis=1).mean())

Codebook shape: (8192, 3584)
Mean L2 norm (should be ~1): 1.0


In [13]:
d = codebook.shape[1] # Current: 2048

# FAISS search (L2 is equivalent to cosine for unit vectors)
index = faiss.IndexFlatL2(d)
index.add(codebook)

In [15]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = 'alxxtexxr/BIVR-Data'
allow_dir = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

Fetching 105 files:   0%|          | 0/105 [00:00<?, ?it/s]

model_unsloth_Qwen2.5-VL-7B-Instruct-bnb(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Vision data downloaded to: /content/data/model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_jxie_coco_captions_train


In [16]:
import numpy as np
import faiss

# Load all batch file paths
batch_paths = sorted([f for f in vision_data_dir.glob('*.npz')])
batch_size = len(train_dataset) // len(batch_paths) # Current: 100

distances = []
codebook_idxs = []

for batch_path in batch_paths:
    batch = np.load(batch_path, mmap_mode='r')
    visual_embs_i = batch['visual_embs'].astype(np.float32) # Current: (32_400, 2048)
    
    faiss.normalize_L2(visual_embs_i)
    
    distances_i, codebook_idxs_i = index.search(visual_embs_i, 1)
    codebook_idxs_i = codebook_idxs_i.squeeze()
    
    distances_i = distances_i.flatten()                       # Current: (32400,)
    codebook_idxs_i = codebook_idxs_i.reshape(batch_size, -1) # Current: (100, 324)
    
    print(f"Distances shape: {distances_i.shape}, type: {type(distances_i)}")
    print(f"Codebook indices shape: {codebook_idxs_i.shape}, type: {type(codebook_idxs_i)}")
    print()
    
    distances.extend(distances_i)
    codebook_idxs.extend(codebook_idxs_i)

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,), type: <class 'numpy.ndarray'>
Codebook indices shape: (12, 270), type: <class 'numpy.ndarray'>

Distances shape: (3240,)

In [17]:
# Compute RMSE
rmse = np.sqrt(np.vstack(distances).mean())
print("RMSE:", rmse)

RMSE: 0.65748596


In [23]:
codebook_idxs[123]

array([6856, 6856, 6856,  955,  527, 2406, 4064, 7039, 4064,  530, 4064,
       8151, 7999, 7999, 8151, 8151, 8151, 8151, 6856, 6856, 6856, 6850,
       5766,  955, 7989, 6667, 7999, 7999, 1006, 8151, 7999, 7999, 2194,
       1006,  776, 6509, 6856, 6856, 6856, 5605, 6856, 3070,  941, 6667,
       4064, 7999, 2697,  713, 7999, 7999, 1006, 1006, 2037, 8151, 6856,
       6856, 6856, 7989, 2544, 6856, 6896, 7989,  776, 7999, 7999, 8151,
       3126, 7999, 7999, 5766, 7999, 5605, 2412, 6856, 6856, 7986,  955,
       6856,  955,  941,  955,  527,  917, 3825, 7999, 7999, 7999, 7999,
       7999, 6110, 5766, 6856, 6856,  955, 6856, 2929, 6856, 7989,  955,
       2963, 2963, 1006, 7999, 7999, 7999, 7251, 1006, 6110, 3998, 1049,
       2184, 6856, 6856, 6856, 5766, 6856, 5766, 2963, 2490, 3742, 1006,
       7999, 2434, 1006, 1006, 8151, 5766, 5331, 6878, 6856,  955, 6856,
       5828, 7989, 7989, 7989, 2963, 4832, 3345, 1006, 7999, 1006, 7999,
       8151, 3998, 1049,  955, 1072, 6856, 6856, 68

In [24]:
def serialize_img_data(example, idx):
    codebook_idxs_i = codebook_idxs[idx].tolist()
    image_serialized = "".join([f'<|vtoken_{i}|>' for i in codebook_idxs_i])
    example['image_serialized'] = image_serialized
    example['codebook_idxs'] = codebook_idxs_i
    return example

dataset_serialized = train_dataset.map(serialize_img_data, with_indices=True)

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [29]:
dataset_serialized.push_to_hub(f'alxxtexxr/COCO-Captions-{str((TRAIN_SIZE+TEST_SIZE)/1000)}K-Serialized')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   7%|6         | 32.1MB /  475MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/COCO-Captions-1.2K-Serialized/commit/f519aec8fd3b27965fd15ddc34305777e6e16c21', commit_message='Upload dataset', commit_description='', oid='f519aec8fd3b27965fd15ddc34305777e6e16c21', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/COCO-Captions-1.2K-Serialized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/COCO-Captions-1.2K-Serialized'), pr_revision=None, pr_num=None)